In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import xarray as xr

from esdglider import defaults, gcp, paths, plots, utils

deployment_name = "chovy-20261005"; mode = "delayed"

home = Path.home()
glider_paths = paths.get_path_glider(
    deployment_name = deployment_name, 
    mode = mode, 
    home_path = home,
)
gcp.gcs_mount_bucket_check(defaults.BUCKET_DATA_OUT, glider_paths["data_out_path"], ro=True)

In [ ]:
ds_sci = xr.load_dataset(glider_paths["tsscipath"])
# df_sci = ds_sci.to_pandas()
ds_sci

In [ ]:
ts = plots.ts_plot("depth", ds_sci, show=True)

In [ ]:
def ts_plot_plotly(
    var: str,
    ds: xr.Dataset,
    *,
    salinity_var: str = "salinity",
    temp_var: str = "potential_temperature",
    time_var: str = "time",
    profile_var: str = "profile_index",
) -> go.Figure | None:
    """Create an interactive Plotly T-S plot showing Time, Profile Index, and Variable on hover."""
    if var not in ds or var in [temp_var, profile_var]:
        # _log.info(f"Skipping variable {var}")
        return None

    # _log.info(f"Making Plotly TS plot for variable {var}")
    deployment_name = utils.get_deployment_name(ds)
    start, end = utils.get_date_start_end(ds)

    # Calculate density grid contours
    Sg, Tg, sigma = utils.calc_ts(ds)

    fig = go.Figure()

    # 1. Isopycnal (Density) Contour Lines
    x_grid = Sg[0, :] if Sg.ndim == 2 else Sg
    y_grid = Tg[:, 0] if Tg.ndim == 2 else Tg

    fig.add_trace(
        go.Contour(
            x=x_grid,
            y=y_grid,
            z=sigma,
            showscale=False,
            contours={
                "coloring": "none",
                "showlabels": True,
                "labelfont": {"size": 10, "color": "grey"},
            },
            line={"color": "grey", "width": 1},
            hoverinfo="z",
            name="Density (\u03c3)",
        )
    )

    # 2. Extract Data & Formatted Custom Hover Attributes
    col_data = plots._adj_var(ds, var)
    col_label = plots._adj_var_label(ds, var)

    # Format timestamps
    time_str = pd.to_datetime(ds[time_var].values).strftime("%Y-%m-%d %H:%M:%S")

    # Extract profile index if available (fill with 'N/A' if missing)
    if profile_var in ds:
        profiles = ds[profile_var].values
    else:
        profiles = ["N/A"] * len(time_str)

    # Stack time and profile index together into customdata
    # %{customdata[0]} = Time, %{customdata[1]} = Profile Index
    custom_hover_data = list(zip(time_str, profiles))

    # 3. Glider Observations Scatter
    fig.add_trace(
        go.Scatter(
            x=ds[salinity_var].values,
            y=ds[temp_var].values,
            mode="markers",
            customdata=custom_hover_data,
            marker={
                "size": 4,
                "color": col_data,
                "colorscale": "Viridis",
                "colorbar": {"title": col_label},
                "showscale": True,
            },
            hovertemplate=(
                "<b>Time</b>: %{customdata[0]}<br>"
                "<b>Profile Index</b>: %{customdata[1]}<br>"
                f"<b>{col_label}</b>: %{{marker.color:.2f}}<extra></extra>"
            ),
            name=var,
        )
    )

    # 4. Layout & Formatting
    fig.update_layout(
        title=f"{deployment_name} from {start} to {end}",
        xaxis_title=plots._adj_var_label(ds, salinity_var),
        yaxis_title=plots._adj_var_label(ds, temp_var),
        template="plotly_white",
        width=850,
        height=700,
    )

    return fig

In [ ]:
ts_plot_plotly("depth", ds_sci)